# Virtual "Air" Mouse Using MediaPipe
    1. This script implements a virtual mouse using a desktop's front-facing camera and the MediaPipe framework to track finger movements and poses.
    2. The air mouse controls consists of
        - Mouse Cursor Navigation: Index Finger Point
        - Left-Click: Index Finger to Thumb Finger
        - Right-Click: Middle Finger to Thumb Finger
EE146, W26 Project \
Authors: Emanual Charro, Angel Pulido

In [ ]:
# 1. Uninstall the broken versions
!pip uninstall -y mediapipe mediapipe-model-maker

# 2. Reinstall, explicitly pinning mediapipe to the stable 0.10.14 version
!pip install -q --no-cache-dir "numpy<2.0.0" "tensorflow==2.15.0" "mediapipe==0.10.14" mediapipe-model-maker

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 61.0/61.0 kB 75.9 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 43.6/43.6 kB 50.0 MB/s eta 0:00:00
  Preparing metadata (setup.py) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 475.3/475.3 MB 169.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 35.7/35.7 MB 218.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 18.3/18.3 MB 201.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 133.3/133.3 kB 222.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.7/1.7 MB 265.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.0/1.0 MB 343.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 294.9/294.9 kB 276.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 5.5/5.5 MB 186.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 442.0/442.0 kB 311.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 241.2/241.2 kB 305.4 MB/s eta 0:00:0

In [ ]:
# Download MediaPipe Hand LandMarker Model
!wget -q https://storage.googleapis.com/mediapipe-models/hand_landmarker/hand_landmarker/float16/1/hand_landmarker.task

# Finger Pose Evaluation

## Develop Custom Model

### Configuration

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

ValueError: mount failed

In [ ]:
import os
import tensorflow as tf
from mediapipe_model_maker import gesture_recognizer
from google.colab import files

In [ ]:
# Define paths for your specific dataset
zip_path = "/content/drive/MyDrive/EE146 Project - Virtual Mouse/v_mouse_sample.zip"
dataset_path = "/content/v_mouse_dataset"

# Extract the dataset
if not os.path.exists(dataset_path):
    !unzip -q "{zip_path}" -d "{dataset_path}"
    print("Dataset extracted.")

# Verify the labels
labels = []
for i in os.listdir(dataset_path):
    if os.path.isdir(os.path.join(dataset_path, i)):
        labels.append(i)
print("Detected gesture labels:", labels)

### Training/Testing

In [ ]:
# Load the dataset and extract hand landmarks
data = gesture_recognizer.Dataset.from_folder(
    dirname=dataset_path,
    hparams=gesture_recognizer.HandDataPreprocessingParams()
)

In [ ]:
# Split the data: 80% for training, 10% for validation, and 10% for testing
train_data, rest_data = data.split(0.8)
validation_data, test_data = rest_data.split(0.5)

In [ ]:
# Configure training parameters
# Epochs can be changed to 20 or 30 to train the model longer
hparams = gesture_recognizer.HParams(export_dir="v_mouse_model", epochs=20)
options = gesture_recognizer.GestureRecognizerOptions(hparams=hparams)

In [ ]:
# Train the custom model
print("Starting training...")
model = gesture_recognizer.GestureRecognizer.create(
    train_data=train_data,
    validation_data=validation_data,
    options=options
)

#### Evaluate Model

In [ ]:
# Evaluate the model on the unseen test dataset
loss, acc = model.evaluate(test_data, batch_size=1)
print(f"\nTest loss: {loss:.4f}, Test accuracy: {acc:.4f}")

### Export and Download Model

In [ ]:
# Export the finalized model
model.export_model()
print("Model exported successfully.")

# Automatically trigger the download of your custom .task file
files.download('v_mouse_model/gesture_recognizer.task')

## Evaluation

In [ ]:
import mediapipe as mp
import numpy as np
from mediapipe.tasks import python
from mediapipe.tasks.python import vision
from pathlib import Path
import matplotlib.pyplot as plt
from sklearn.metrics import confusion_matrix, ConfusionMatrixDisplay, classification_report

### Configuration

In [ ]:
# Point to the unzipped dataset
base_dir = Path("/content/v_mouse_dataset")

# Define the paths to class folders
dataset_paths = {
    "left-click":  base_dir / "left-click",
    "point":       base_dir / "point",
    "right-click": base_dir / "right-click"
}

# Point to the custom model
model_path = '/content/v_mouse_model/gesture_recognizer.task'

try:
    base_options = python.BaseOptions(model_asset_path=model_path)
    options = vision.GestureRecognizerOptions(base_options=base_options)
    recognizer = vision.GestureRecognizer.create_from_options(options)
    print("Loaded custom model.")
except Exception as e:
    print(f"Error loading model: {e}")
    exit()

y_true = []
y_pred = []

### Process

In [ ]:
import numpy as np

print("Evaluating test dataset...")

y_true = []
y_pred = []

# Get the mapping of index numbers to string names (e.g., 0 -> "left-click")
label_names = test_data.label_names

for feature_tensor, label_tensor in test_data.gen_tf_dataset(batch_size=1):

    # 1. Get the True Label
    # We convert the TensorFlow tensor to a numpy array, then find the one-hot index
    true_index = np.argmax(label_tensor.numpy()[0])
    true_label_name = label_names[true_index]

    # 2. Get the Prediction
    # Because Model Maker creates a Keras model under the hood, we can access
    # it via '._model' to predict directly on the pre-processed landmark tensors.
    # verbose=0 stops it from printing a progress bar for every single image
    prediction_result = model._model.predict(feature_tensor, verbose=0)

    pred_index = np.argmax(prediction_result[0])
    pred_label_name = label_names[pred_index]

    y_true.append(true_label_name)
    y_pred.append(pred_label_name)

print(f"Processed {len(y_true)} images as [true].")
print(f"Processed {len(y_pred)} images as [predicted].")

### Generate Metrics and Plots

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from sklearn.metrics import classification_report, confusion_matrix, ConfusionMatrixDisplay

# 1. Force Colab to render plots inline
%matplotlib inline

labels = ["point", "left-click", "right-click", "none"]

print("\n" + "="*60)
print("Model Classification Summary")
print("="*60)

if len(y_true) > 0:

    cm = confusion_matrix(y_true, y_pred, labels=labels)

    print(classification_report(y_true, y_pred, labels=labels, zero_division=0))

    # Clear any hanging plots in the background
    plt.clf()
    plt.close('all')

    fig, ax = plt.subplots(figsize=(8, 8))
    disp = ConfusionMatrixDisplay(confusion_matrix=cm, display_labels=labels)

    # Try plotting without extra formatting first
    disp.plot(ax=ax, cmap='Blues', colorbar=False)
    plt.title('Virtual Mouse Gesture Confusion Matrix')

    # Explicitly call show
    plt.show()
else:
    print("No images were processed. Please verify your dataset_paths.")

### Display Intermediate Images

In [ ]:
import cv2
import random
import mediapipe as mp
from mediapipe.tasks import python
from mediapipe.tasks.python import vision
from mediapipe.framework.formats import landmark_pb2
from pathlib import Path
import matplotlib.pyplot as plt

In [ ]:
# Load MediaPipe drawing tools for the "skeleton"
mp_drawing = mp.solutions.drawing_utils
mp_hands = mp.solutions.hands
mp_drawing_styles = mp.solutions.drawing_styles

# 1. SETUP =================================================
base_dir = Path("/content/v_mouse_dataset")
dataset_paths = {
    "left-click":  base_dir / "left-click",
    "point":       base_dir / "point",
    "right-click": base_dir / "right-click"
}

# Load the custom model
model_path = '/content/v_mouse_model/gesture_recognizer.task'
base_options = python.BaseOptions(model_asset_path=model_path)
options = vision.GestureRecognizerOptions(base_options=base_options)
recognizer = vision.GestureRecognizer.create_from_options(options)

# 2. Visualization Loop =============================================
num_samples_per_class = 3 # Adjust this to see more/fewer images
fig, axes = plt.subplots(len(dataset_paths), num_samples_per_class, figsize=(12, 12))
fig.suptitle("Original Images & Extracted Skeletons", fontsize=16)

for row_idx, (label, folder_path) in enumerate(dataset_paths.items()):
    if not folder_path.exists():
        continue

    # Grab all valid images and select random samples
    all_images = [f for f in folder_path.iterdir() if f.suffix.lower() in ['.png', '.jpg', '.jpeg']]
    samples = random.sample(all_images, min(num_samples_per_class, len(all_images)))

    for col_idx, file_path in enumerate(samples):
        ax = axes[row_idx, col_idx]

        # Load the image
        mp_image = mp.Image.create_from_file(str(file_path))

        # Create a writable copy of the image array so we can draw on it
        cv2_img = mp_image.numpy_view().copy()

        # Process the image through the custom AI
        result = recognizer.recognize(mp_image)
        prediction = "none"

        # If a hand is detected, extract the data and draw the skeleton
        if result.hand_landmarks:
            # 1. Get the gesture prediction
            if result.gestures and len(result.gestures[0]) > 0:
                prediction = result.gestures[0][0].category_name

            # 2. Convert the Task API landmarks into the older Protobuf format
            #    required by the drawing utility
            hand_landmarks_proto = landmark_pb2.NormalizedLandmarkList()
            hand_landmarks_proto.landmark.extend([
                landmark_pb2.NormalizedLandmark(x=lm.x, y=lm.y, z=lm.z)
                for lm in result.hand_landmarks[0]
            ])

            # 3. Draw the joints and connecting lines directly onto the image
            mp_drawing.draw_landmarks(
                cv2_img,
                hand_landmarks_proto,
                mp_hands.HAND_CONNECTIONS,
                mp_drawing_styles.get_default_hand_landmarks_style(),
                mp_drawing_styles.get_default_hand_connections_style()
            )

        # Convert from BGR (OpenCV format) to RGB (Matplotlib format)
        rgb_img = cv2.cvtColor(cv2_img, cv2.COLOR_BGR2RGB)

        # Plot the annotated image
        ax.imshow(rgb_img)

        # Color the title text green if correct, red if incorrect (or 'none')
        title_color = 'green' if prediction == label else 'red'
        ax.set_title(f"True: {label}\nPred: {prediction}", color=title_color, fontsize=12)
        ax.axis('off')

plt.tight_layout()
plt.subplots_adjust(top=0.90) # Make room for the main title
plt.show()

### Plot Multiclass ROC Curve

In [ ]:
import mediapipe as mp
import numpy as np
import matplotlib.pyplot as plt
from mediapipe.tasks import python
from mediapipe.tasks.python import vision
from pathlib import Path
from sklearn.metrics import roc_curve, auc
from sklearn.preprocessing import label_binarize

# 1. SETUP =========================================
base_dir = Path("/content/v_mouse_dataset")
dataset_paths = {
    "left-click":  base_dir / "left-click",
    "point":       base_dir / "point",
    "right-click": base_dir / "right-click"
}

model_path = '/content/v_mouse_model/gesture_recognizer.task'
base_options = python.BaseOptions(model_asset_path=model_path)
options = vision.GestureRecognizerOptions(base_options=base_options)
recognizer = vision.GestureRecognizer.create_from_options(options)

# All possible output classes
labels = ["point", "left-click", "right-click", "none"]

y_true = []
y_scores = [] # Store probability vectors

# 2. PROCESSING LOOP (Extracting Probabilities) =================
print("Extracting confidence scores for ROC...")

for true_label, folder_path in dataset_paths.items():
    if not folder_path.exists():
        continue

    for file_path in folder_path.iterdir():
        if file_path.suffix.lower() not in ['.png', '.jpg', '.jpeg']:
            continue

        mp_image = mp.Image.create_from_file(str(file_path))
        result = recognizer.recognize(mp_image)

        y_true.append(true_label)

        # Initialize a probability vector of zeros: [0.0, 0.0, 0.0, 0.0]
        prob_vector = [0.0] * len(labels)

        if result.gestures and len(result.gestures[0]) > 0:
            top_gesture = result.gestures[0][0]
            pred_name = top_gesture.category_name
            score = top_gesture.score # The raw confidence floating point

            # Apply the confidence score to the predicted class
            if pred_name in labels:
                idx = labels.index(pred_name)
                prob_vector[idx] = score
        else:
            # If nothing is detected, the AI is 100% confident it is "none"
            idx = labels.index("none")
            prob_vector[idx] = 1.0

        y_scores.append(prob_vector)

print("Plotting ROC Curves...")

# 3. PLOT MULTICLASS ROC CURVE ================================
# Binarize the true labels for One-vs-Rest comparison
y_true_bin = label_binarize(y_true, classes=labels)
y_scores = np.array(y_scores)

fig, ax = plt.subplots(figsize=(10, 8))

# Plot an ROC curve for each gesture
colors = ['blue', 'green', 'red', 'gray']
for i, class_name in enumerate(labels):
    # Calculate False Positive Rate and True Positive Rate
    fpr, tpr, _ = roc_curve(y_true_bin[:, i], y_scores[:, i])
    roc_auc = auc(fpr, tpr)

    ax.plot(fpr, tpr, color=colors[i], lw=2,
            label=f'{class_name} (AUC = {roc_auc:.3f})')

# Plot the random guess baseline
ax.plot([0, 1], [0, 1], 'k--', lw=2, label='Random Guess (AUC = 0.500)')

ax.set_xlim([0.0, 1.0])
ax.set_ylim([0.0, 1.05])
ax.set_xlabel('False Positive Rate', fontsize=12)
ax.set_ylabel('True Positive Rate', fontsize=12)
ax.set_title('One-vs-Rest Multiclass ROC Curve for Virtual Mouse', fontsize=14)
ax.legend(loc="lower right", fontsize=11)
ax.grid(alpha=0.3)

plt.tight_layout()
plt.show()

# Virtual Mouse Implementation

***Run test script `virtual_mouse_test.py`***